In [ ]:
!pwd

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("AAI_API_KEY")

In [ ]:
import assemblyai as aai
aai.settings.api_key = api_key

In [ ]:
from pathlib import Path
import requests
import time

base_dir = Path.cwd()

transaction_dir = base_dir / "Unclipped Processed Transactions"

In [ ]:
files = [f for f in transaction_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

In [ ]:
base_url = "https://api.assemblyai.com"

headers = {
"authorization": api_key
}

In [ ]:
#Collect Transcripts (no timestamps) for each file in files
#Enforce 2 Speakers

transcripts = {}

for file in files:
    with open(file, "rb") as f:
        response = requests.post(base_url + "/v2/upload", headers=headers, data=f)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, Response: {response.text}")
            response.raise_for_status()
        upload_json = response.json()
        audio_file = upload_json["upload_url"]


    data = {
    "audio_url": audio_file, # You can also use a URL to an audio or video file on the web
    "speech_models": ["universal-3-pro", "universal-2"],
    "language_detection": True,
    "speaker_labels": True,
    "speakers_expected": 2
    }

    response = requests.post(base_url + "/v2/transcript", headers=headers, json=data)
    transcript_id = response.json()["id"]
    polling_endpoint = base_url + f"/v2/transcript/{transcript_id}"

    while True:
        transcript = requests.get(polling_endpoint, headers=headers).json()
        if transcript["status"] == "completed":
            break
        elif transcript["status"] == "error":
            raise RuntimeError(f"Transcription failed: {transcript['error']}")
        else:
            time.sleep(3)

    transcripts[file] = transcript

In [ ]:
#Test to see if just one utterance
one_utterance = []

for file, transcript in transcripts.items():
    if len(transcript["utterances"]) == 1:
        one_utterance.append(file)

In [ ]:
#Test to see if more than two speakers
two_plus_speakers = []

for file, transcript in transcripts.items():
    speaker_set = set()
    for utterance in transcript["utterances"]:
        speaker = utterance["speaker"]
        if speaker not in speaker_set:
            speaker_set.add(speaker)

    if len(speaker_set) > 2:
        two_plus_speakers.append(file)